# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata
print(f"{md.name}: {md.description}")

# Optionally, show key metadata
print(f"\nVersion: {md.version}\nPublished: {md.datePublished}")
print(f"License: {md.license}")
print(f"Cite as: {md.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all record sets in the dataset, each with its `@id`, referencable title, and its available fields (and their `@id`s).

In [ ]:
# List record sets available in the dataset
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the Croissant metadata (metadata does not include cr:recordSet links). Trying to enumerate from DataDownload objects.")
    # Attempt to infer record set IDs from distribution
    from pprint import pprint
    pprint(md.to_json().get('distribution', []))
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  name: {getattr(rs, 'name', 'N/A')}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id}, name: {getattr(field, 'name', 'N/A')}")
        print('-'*40)

# For demonstration, also print names of all record sets by @id
record_set_ids = [rs.id for rs in record_sets] if record_sets else []
print("Found record sets:", record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If record sets are not explicitly in the metadata's `recordSet`, we'll attempt to load from the primary record set if available, or try to find one.

In [ ]:
# Use the list of record set IDs from the previous overview
if not record_set_ids:
    # If not found, try to use the default (often "Main" or similar)
    # We'll attempt to use a dummy record set id or skip
    print("No record sets to load.")
else:
    dataframes = {}
    for rs_id in record_set_ids:
        print(f"Loading records from RecordSet @{rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for {rs_id}")
            print("Columns:", df.columns.tolist())
            print(df.head())
        else:
            print(f"No records extracted for {rs_id}.")
    if dataframes:
        # Choose the first found record set for the next steps
        primary_rs_id = list(dataframes.keys())[0]
        primary_df = dataframes[primary_rs_id]
        print("\nPrimary record set selected for analysis:", primary_rs_id)
        print(primary_df.head())
    else:
        print("Could not extract records for any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

You must use the actual field `@id` strings from the record set for column-based operations. Modify the field and group names below to use the correct ones from the dataset (e.g., 'gad7_total_score', 'phq9_total_score', etc, as per field `@id`).

In [ ]:
# For demonstration, we pick a typical field @id for a numeric analysis
# Replace '<numeric_field_id>' and '<group_field_id>' below with actual @id values from cell above

# Set up the analysis variables
if dataframes:
    numeric_field = None
    group_field = None
    rs_df = primary_df
    # Try to select "gad7_total_score" or similar field by column name (which will use the @id or a slug)
    # Print all columns for review
    print("Columns in selected DataFrame:", rs_df.columns.tolist())
    # Heuristic: pick a field that looks like a score or numeric value
    for col in rs_df.columns:
        if 'gad7' in col.lower() and 'score' in col.lower():
            numeric_field = col
            break
    if not numeric_field:
        # Try PHQ9, PSQ, fallback to integer fields
        for col in rs_df.columns:
            if ('phq' in col.lower() or 'psq' in col.lower()) and 'score' in col.lower():
                numeric_field = col
                break
    if not numeric_field:
        for col in rs_df.columns:
            if rs_df[col].dtype in [int, float]:
                numeric_field = col
                break

    # Now pick a grouping field: try 'gender' or a demographic attribute
    for col in rs_df.columns:
        if 'gender' in col.lower():
            group_field = col
            break
    if not group_field:
        for col in rs_df.columns:
            if ('age' in col.lower() or 'education' in col.lower()) and rs_df[col].dtype == object:
                group_field = col
                break

    print(f"Numeric field used: {numeric_field}")
    print(f"Group field used: {group_field}")

    # EDA only if fields found
    if numeric_field and numeric_field in rs_df.columns:
        # Remove obviously missing values and filter for score > threshold
        threshold = 5
        filtered_df = rs_df[[numeric_field] + ([group_field] if group_field else [])].copy()
        filtered_df = filtered_df[filtered_df[numeric_field].apply(pd.to_numeric, errors='coerce').notnull()]
        filtered_df[numeric_field] = filtered_df[numeric_field].astype(float)
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field if available
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("Could not find a suitable numeric field for analysis.")
else:
    print("No DataFrames available from extraction step.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib/seaborn to plot the distribution of the selected numeric field, or the grouped means by demographic attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and numeric_field in primary_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(primary_df[numeric_field].dropna().astype(float), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in primary_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=primary_df[group_field], y=primary_df[numeric_field].astype(float), showfliers=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("Cannot plot: numeric field or DataFrame not available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration. For example:
- The dataset provides a rich array of mental health survey responses, including GAD-7, PHQ-9, and PSQ scores.
- Demographic grouping (e.g., by gender) can reveal differences in psychological symptom distributions.
- The dataset is suitable for further machine learning and statistical analysis on mental health indicators in Kilifi County, Kenya.

Next steps could include modeling, more advanced visualizations, or exploring field-level semantics using the field/column `@id` values for reproducible pipelines.